In [1]:
from dataclasses import dataclass

In [24]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    raw_data_file: Path
    ticker: str
    start_date: str
    end_date: str
    

In [4]:
!pwd
import os
# os.chdir("../")

/c/Users/Ibk/Desktop/data project/Stock-Price-Prediction-MLOps


In [19]:
from src.stock_prediction.utils.common import *
read_yaml(CONFIG_FILE_PATH)

2026-08-07 07:36:15,455 | INFO | common| YAML file: config\config.yaml loaded successfully.


ConfigBox({'artifacts_root': 'artifacts', 'data_ingestion': {'root_dir': 'artifacts/data_ingestion', 'raw_data_file': 'artifacts/data_ingestion/stock_data.csv', 'ticker': 'AAPL', 'start_date': datetime.date(2000, 1, 1), 'end_date': datetime.date(2026, 8, 4)}})

In [25]:
from stock_prediction.constants import *
from src.stock_prediction.utils.common import *
class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])
    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        
        create_directories([config.root_dir])
        
        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            raw_data_file=Path(config.raw_data_file),
            ticker=config.ticker,
            start_date=config.start_date,
            end_date=config.end_date
        )
        return data_ingestion_config

In [26]:
a = ConfigurationManager()
a.get_data_ingestion_config()

2026-08-07 07:50:50,891 | INFO | common| YAML file: config\config.yaml loaded successfully.
2026-08-07 07:50:50,893 | INFO | common| YAML file: params.yaml loaded successfully.
2026-08-07 07:50:50,895 | INFO | common| Directory created at: artifacts
2026-08-07 07:50:50,897 | INFO | common| Directory created at: artifacts/data_ingestion


DataIngestionConfig(root_dir='artifacts/data_ingestion', raw_data_file=WindowsPath('artifacts/data_ingestion/stock_data.csv'), ticker='AAPL', start_date=datetime.date(2000, 1, 1), end_date=datetime.date(2026, 8, 4))

In [13]:
import yfinance as yf
yf.download?

Signature:
yf.download(
    tickers,
    start=None,
    end=None,
    actions=False,
    threads=True,
    ignore_tz=None,
    group_by='column',
    auto_adjust=True,
    back_adjust=False,
    repair=False,
    keepna=False,
    progress=True,
    period='1mo if start & end None',
    interval='1d',
    prepost=False,
    rounding=False,
    timeout=10,
    session=None,
    multi_level_index=True,
) -> Optional[pandas.core.frame.DataFrame]
Docstring:
Download yahoo tickers
:Parameters:
    tickers : str, list
        List of tickers to download
    period : str
        Valid periods: 1d,5d,1mo,3mo,6mo,1y,2y,5y,10y,ytd,max
        Default: '1mo' if start & end None
        Either Use period parameter or use start and end
    interval : str
        Valid intervals: 1m,2m,5m,15m,30m,60m,90m,1h,1d,5d,1wk,1mo,3mo
        Intraday data cannot extend last 60 days
    start: str
        Download start date string (YYYY-MM-DD) or _datetime, inclusive.
        Default is 99 years ago
       

In [ ]:
# !pip install yfinance

  Using cached yfinance-1.5.2-py2.py3-none-any.whl.metadata (6.2 kB)
  Using cached multitasking-0.0.13-py3-none-any.whl.metadata (16 kB)
  Using cached peewee-4.3.0-py3-none-any.whl.metadata (10 kB)
  Using cached curl_cffi-0.16.0-cp310-abi3-win_amd64.whl.metadata (17 kB)
Using cached yfinance-1.5.2-py2.py3-none-any.whl (144 kB)
Using cached curl_cffi-0.16.0-cp310-abi3-win_amd64.whl (2.0 MB)
Using cached multitasking-0.0.13-py3-none-any.whl (16 kB)
Using cached peewee-4.3.0-py3-none-any.whl (179 kB)



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [66]:
from src.stock_prediction.utils.common import save_json



In [67]:
import pandas as pd
from pathlib import Path
def save_csv(data: pd.DataFrame, file_path: Path) -> None:
    """Saves the DataFrame to a CSV file.

    Args:
        data (pd.DataFrame): The DataFrame to save.
        file_path (Path): The path where the CSV file will be saved.
    """
    try:
        data.to_csv(file_path, index=False)
        logger.info(f"Data saved to {file_path}")
    except Exception as e:
        logger.error(f"Error saving data to {file_path}: {e}")
        raise e

In [27]:
import yfinance as yf
from stock_prediction import logger
import pandas as pd
from src.stock_prediction.entity.config_entity import DataIngestionConfig
from src.stock_prediction.utils.common import *

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config =  config
    def fetch_file(self):
        try:
            
            logger.info(f"Data downloading for ticker: {self.config.ticker} from \
                {self.config.start_date} to {self.config.end_date}")
            data = yf.download(
                tickers=self.config.ticker,
                start=self.config.start_date,
                end=self.config.end_date
            )
            logger.info(f"Data downloaded successfully for ticker: {self.config.ticker} from \
                {self.config.start_date} to {self.config.end_date}")
            if not (data.empty):
                data = self.reset_columns(data)           
                save_csv(data, self.config.raw_data_file)

            else:
                logger.error(f"No data found for ticker: {self.config.ticker} from \
                    {self.config.start_date} to {self.config.end_date}") 
                raise ValueError(f"No data found for ticker: {self.config.ticker} from \
                    {self.config.start_date} to {self.config.end_date}")
        except Exception as e:
            logger.error(f"Error while downloading data for ticker: {self.config.ticker} ")
            raise e

        
        
    def reset_columns(self, data: pd.DataFrame) -> pd.DataFrame:
        if isinstance(data.columns, pd.MultiIndex):
            data.columns = data.columns.droplevel(1)
        data = data.reset_index()
        return data

In [28]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(data_ingestion_config)
    data_ingestion.fetch_file()
except Exception as e:
    logger.exception(e)
    raise e

2026-08-07 07:51:05,048 | INFO | common| YAML file: config\config.yaml loaded successfully.
2026-08-07 07:51:05,051 | INFO | common| YAML file: params.yaml loaded successfully.
2026-08-07 07:51:05,053 | INFO | common| Directory created at: artifacts
2026-08-07 07:51:05,055 | INFO | common| Directory created at: artifacts/data_ingestion
2026-08-07 07:51:05,056 | INFO | 2975460377| Data downloading for ticker: AAPL from                 2000-01-01 to 2026-08-04


[*********************100%***********************]  1 of 1 completed

2026-08-07 07:51:05,234 | INFO | 2975460377| Data downloaded successfully for ticker: AAPL from                 2000-01-01 to 2026-08-04
2026-08-07 07:51:05,284 | INFO | common| Data saved to artifacts\data_ingestion\stock_data.csv


In [16]:
type(data_ingestion_config.start_date)

datetime.date

In [30]:
type(data.columns)

pandas.core.indexes.multi.MultiIndex

In [41]:
data.head(1)

Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,
2020-01-02,72.333862,72.39407,71.091169,71.344039,135480400


In [39]:
data.columns.droplevel(1)

Index(['Close', 'High', 'Low', 'Open', 'Volume'], dtype='object', name='Price')

In [48]:
data.reset_index()

Price,Date,Close,High,Low,Open,Volume
Ticker,,AAPL,AAPL,AAPL,AAPL,AAPL
0,2020-01-02,72.333862,72.394070,71.091169,71.344039,135480400
1,2020-01-03,71.630638,72.389257,71.406666,71.563205,146322800
2,2020-01-06,72.201393,72.239927,70.503531,70.753999,118387200
3,2020-01-07,71.861839,72.466322,71.642681,72.211041,108872000
4,2020-01-08,73.017838,73.318877,71.565621,71.565621,132079200
...,...,...,...,...,...,...
751,2022-12-23,129.659424,130.210076,127.476472,128.735109,63814900
752,2022-12-27,127.859932,129.216906,126.571797,129.187408,69007800
753,2022-12-28,123.936531,128.843251,123.769370,127.505948,85438400


In [32]:
import pandas as pd
pd.MultiIndex

pandas.core.indexes.multi.MultiIndex

In [ ]:
Configuration

In [ ]:
class DataIngestion:
    def __init__(self, config)

In [ ]:
from stock_prediction.config.configuration import ConfigurationManager
from stock_prediction.components.data_ingestion import DataIngestion
from stock_prediction import logger


STAGE_NAME = "Data Ingestion"


class DataIngestionTrainingPipeline:
    def __init__(self):
        pass

    def main(self):
        config = ConfigurationManager()
        data_ingestion_config = config.get_data_ingestion_config()
        data_ingestion = DataIngestion(config=data_ingestion_config)
        data_ingestion.fetch_file()


if __name__ == "__main__":
    try:
        logger.info(f">>>>>> stage {STAGE_NAME} started <<<<<<")
        obj = DataIngestionTrainingPipeline()
        obj.main()
        logger.info(f">>>>>> stage {STAGE_NAME} completed <<<<<<\n\nx==========x")
    except Exception as e:
        logger.exception(e)
        raise e